# Group 4 — Fun Finder Pittsburgh 🎉
### CS 1656 Final Project | Combined Analysis

| Member | Sub-metric | Dataset |
|---|---|---|
| Sachit Anand | Parks | City of Pittsburgh Parks |
| Rhonda Ojongmboh | Playgrounds | City of Pittsburgh Playgrounds |
| Osarumen Okhiku | Courts and Rinks | City of Pittsburgh Courts and Rinks |

---
## 1. Introduction

What makes a neighborhood fun? Our group agreed that fun should be defined in a way that is tangible, measurable, and fair — not based on nightlife or restaurant counts, which heavily favor wealthier or more commercial neighborhoods, but on free public recreational infrastructure that every resident can access regardless of income.

We settled on three sub-metrics, each sourced from the [Western Pennsylvania Regional Data Center (WPRDC)](https://www.wprdc.org/):

- **Parks** (Sachit) — green outdoor spaces for relaxation and unstructured recreation
- **Playgrounds** (Rhonda) — purpose-built active play equipment in city parks
- **Courts and Rinks** (Osarumen) — sport-specific facilities for basketball, tennis, hockey, and more

Together, these three datasets capture a neighborhood's total commitment to public recreational fun. A neighborhood that scores highly across all three is one where residents truly have the most options to go outside, be active, and enjoy their community.

Each member analyzed their dataset individually and produced a normalized sub-score. In this notebook we combine those scores into a single **Fun Score** and name the most fun neighborhood in Pittsburgh.

---
## 2. The Metric

**Fun Score** = (park_score + playground_score + courts_score) ÷ 3

Each sub-score is the neighborhood's count of that feature divided by the maximum count across all neighborhoods, giving a value between 0 and 1. This normalization ensures no single sub-metric dominates the final score just because its raw numbers happen to be larger.

A neighborhood that leads in all three categories receives a perfect score of 1.0. A neighborhood absent from all three datasets receives 0.0.

| Sub-metric | Formula | Max raw value | Neighborhood with max |
|---|---|---|---|
| park_score | park_count / 12 | 12 parks | East Liberty |
| playground_score | playground_count / 8 | 8 playgrounds | Squirrel Hill South |
| courts_score | courts_count / 23 | 23 courts/rinks | Squirrel Hill South |

---
## 3. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

os.makedirs('data', exist_ok=True)
print('Libraries loaded successfully.')

---
## 4. Load All Three Datasets

In [ ]:
parks       = pd.read_csv('data/parks.csv')
playgrounds = pd.read_csv('data/playgrounds.csv')
courts      = pd.read_csv('data/courts_rinks.csv')

print(f'Parks dataset:       {len(parks)} rows, {parks["neighborhood"].nunique()} unique neighborhoods')
print(f'Playgrounds dataset: {len(playgrounds)} rows, {playgrounds["neighborhood"].nunique()} unique neighborhoods')
print(f'Courts/Rinks dataset:{len(courts)} rows, {courts["neighborhood"].nunique()} unique neighborhoods')

---
## 5. Compute Each Sub-metric

In [ ]:
# ── Sachit: Parks ─────────────────────────────────────────────────────────────
parks_count = (
    parks.dropna(subset=['neighborhood'])
    .groupby('neighborhood').size()
    .reset_index(name='park_count')
)
parks_count['park_score'] = parks_count['park_count'] / parks_count['park_count'].max()

print('Parks — top 5:')
print(parks_count.sort_values('park_count', ascending=False).head().to_string(index=False))

In [ ]:
# ── Rhonda: Playgrounds ───────────────────────────────────────────────────────
pg_count = (
    playgrounds.dropna(subset=['neighborhood'])
    .groupby('neighborhood').size()
    .reset_index(name='playground_count')
)
pg_count['playground_score'] = pg_count['playground_count'] / pg_count['playground_count'].max()

print('Playgrounds — top 5:')
print(pg_count.sort_values('playground_count', ascending=False).head().to_string(index=False))

In [ ]:
# ── Osarumen: Courts and Rinks (active only) ──────────────────────────────────
courts_active = courts[courts['inactive'] == 'f']
courts_count = (
    courts_active.dropna(subset=['neighborhood'])
    .groupby('neighborhood').size()
    .reset_index(name='courts_count')
)
courts_count['courts_score'] = courts_count['courts_count'] / courts_count['courts_count'].max()

print('Courts/Rinks — top 5:')
print(courts_count.sort_values('courts_count', ascending=False).head().to_string(index=False))

---
## 6. Merge and Compute the Fun Score

In [ ]:
# Outer join so every neighborhood from every dataset is included
merged = (
    parks_count[['neighborhood', 'park_count', 'park_score']]
    .merge(pg_count[['neighborhood', 'playground_count', 'playground_score']],
           on='neighborhood', how='outer')
    .merge(courts_count[['neighborhood', 'courts_count', 'courts_score']],
           on='neighborhood', how='outer')
)

# Neighborhoods absent from a category get 0 (no recorded facilities)
merged = merged.fillna(0)

# Fun Score = average of the three normalized sub-scores
merged['fun_score'] = (
    merged['park_score'] + merged['playground_score'] + merged['courts_score']
) / 3

merged = merged.sort_values('fun_score', ascending=False).reset_index(drop=True)

print('Top 15 neighborhoods by Fun Score:')
print(merged[['neighborhood','park_count','playground_count','courts_count','fun_score']]
      .head(15).to_string(index=False))

---
## 7. Visualizations

In [ ]:
# ── Chart 1: Final Fun Score — Top 15 neighborhoods ──────────────────────────
top15 = merged.head(15)

fig, ax = plt.subplots(figsize=(15, 7))
colors = ['#f1c40f' if i == 0 else '#3498db' for i in range(len(top15))]
bars = ax.bar(top15['neighborhood'], top15['fun_score'],
              color=colors, edgecolor='white', linewidth=0.8)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_title('🎉 Fun Score — Top 15 Pittsburgh Neighborhoods\n(Average of normalized parks + playgrounds + courts/rinks)',
             fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel('Fun Score (0–1)', fontsize=12)
ax.set_xlabel('Neighborhood', fontsize=12)
ax.set_xticklabels(top15['neighborhood'], rotation=45, ha='right', fontsize=10)
ax.set_ylim(0, merged['fun_score'].max() * 1.18)
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#f1c40f', label='🏆 Winner'),
    plt.Rectangle((0,0),1,1, color='#3498db', label='Other Top 15')
], fontsize=10)

plt.tight_layout()
plt.savefig('fun_score_top15.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fun_score_top15.png')

In [ ]:
# ── Chart 2: Grouped bar — compare all three sub-scores, top 10 ───────────────
top10 = merged.head(10)
x = np.arange(len(top10))
w = 0.25

fig, ax = plt.subplots(figsize=(15, 7))
ax.bar(x - w, top10['park_score'],       w, label='Parks (Sachit)',          color='#2ecc71', edgecolor='white')
ax.bar(x,     top10['playground_score'], w, label='Playgrounds (Rhonda)',    color='#e74c3c', edgecolor='white')
ax.bar(x + w, top10['courts_score'],     w, label='Courts/Rinks (Osarumen)', color='#9b59b6', edgecolor='white')

ax.set_title('Normalized Sub-metric Scores by Category — Top 10 Neighborhoods',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Normalized Score (0–1)', fontsize=12)
ax.set_xlabel('Neighborhood', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(top10['neighborhood'], rotation=40, ha='right', fontsize=10)
ax.set_ylim(0, 1.2)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('submetric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: submetric_comparison.png')

In [ ]:
# ── Chart 3: Stacked bar — Fun Score decomposed into 3 contributions ──────────
top10 = merged.head(10).copy()

fig, ax = plt.subplots(figsize=(14, 7))
p1 = top10['park_score']       / 3
p2 = top10['playground_score'] / 3
p3 = top10['courts_score']     / 3

ax.bar(top10['neighborhood'], p1, label='Parks',         color='#2ecc71', edgecolor='white')
ax.bar(top10['neighborhood'], p2, label='Playgrounds',   color='#e74c3c', edgecolor='white', bottom=p1)
ax.bar(top10['neighborhood'], p3, label='Courts/Rinks',  color='#9b59b6', edgecolor='white', bottom=p1+p2)

ax.set_title('Fun Score Composition — Top 10 Neighborhoods',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Fun Score Contribution', fontsize=12)
ax.set_xlabel('Neighborhood', fontsize=12)
ax.set_xticklabels(top10['neighborhood'], rotation=40, ha='right', fontsize=10)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('fun_score_stacked.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fun_score_stacked.png')

---
## 8. The Best Neighborhood

In [ ]:
winner = merged.iloc[0]
runner_up = merged.iloc[1]

print('=' * 58)
print(f'  🏆 THE MOST FUN NEIGHBORHOOD IN PITTSBURGH IS:')
print(f'         {winner["neighborhood"].upper()}')
print('=' * 58)
print(f'  Fun Score:    {winner["fun_score"]:.4f}')
print(f'  Parks:        {int(winner["park_count"])}')
print(f'  Playgrounds:  {int(winner["playground_count"])}')
print(f'  Courts/Rinks: {int(winner["courts_count"])}')
print()
print(f'  Runner-up: {runner_up["neighborhood"]} (Fun Score: {runner_up["fun_score"]:.4f})')
print(f'  Margin of victory: {winner["fun_score"] - runner_up["fun_score"]:.4f}')

**Squirrel Hill South** wins the combined Fun Score by a wide margin with a score of **0.8333** — more than 0.28 points ahead of East Liberty in second place (0.5453). This is not a close race. Squirrel Hill South leads or is near the top in all three sub-metrics:

- **#1 in playgrounds** with 8 (score: 1.000)
- **#1 in courts and rinks** with 23 active facilities (score: 1.000)
- **#6 in parks** with 6 (score: 0.500)

Squirrel Hill South does not just win one category — it dominates two out of three while remaining competitive in the third. No other neighborhood comes close to that combination.

---
## 9. Conclusion

After combining three independently-analyzed datasets of public recreational amenities, our data-driven answer is clear: **Squirrel Hill South** is the most fun neighborhood in Pittsburgh.

Our metric was designed to be fair and democratic — it measures only free, publicly accessible infrastructure, and it weights all three sub-metrics equally so no single category could dominate. Squirrel Hill South earns its win legitimately: it has the most playgrounds in the city (8), the most active courts and rinks (23), and a respectable parks count (6). It is a neighborhood where every resident, regardless of age or athletic preference, has an exceptional number of recreational options within reach.

If we had used only one sub-metric, the answer would have been different each time — East Liberty tops the parks count, and both Squirrel Hill South and Highland Park are strong on courts. The combined metric is what makes the result meaningful, because it rewards balanced investment across multiple types of recreation rather than specialization in just one.

---
### Individual Reflections

**Sachit Anand:** Parks alone pointed me toward East Liberty, which surprised me because I personally associate Squirrel Hill South with being a fun, livable neighborhood. The combined data confirmed my instinct about Squirrel Hill South while also showing me that East Liberty deserves more credit than I gave it. Data can reveal things that personal experience misses, and this project showed me that clearly.

**Rhonda Ojongmboh:** Squirrel Hill South was the clear winner of my sub-metric and the overall winner, which matched my intuition. What surprised me was how far behind some more commercially visible neighborhoods — like the Strip District or Shadyside — fell on these metrics. Fun that is built into the physical infrastructure of a neighborhood is different from fun that comes from bars and restaurants, and I think our metric captures something important and underappreciated.

**Osarumen Okhiku:** The courts and rinks data was the most decisive — Squirrel Hill South had more than double the courts of most neighborhoods. What I found interesting was looking at the type breakdown: it is not just basketball courts, it has tennis, hockey rinks, bocce, volleyball, and more. That diversity means the neighborhood serves a much wider range of residents than somewhere with only one type of facility. That kind of variety is, to me, the real definition of a fun neighborhood.